In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# Cell 1: setup
!pip install -q sentencepiece protobuf

from pathlib import Path

DATA_PATH = Path("/kaggle/input/datasets/sqizeeeeeeeee/dataset-llm-finetuning-bt/train_bt.parquet")
WORK_DIR = Path("/kaggle/working")
OUTPUT_DIR = WORK_DIR / "res"
# RESUME_FROM = None
RESUME_FROM = Path("/kaggle/input/datasets/sqizeeeeeeeee/checkpoint1/res")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_PATH.exists(), f"dataset not found at {DATA_PATH}, check the exact input folder name"
print(f"[setup] data: {DATA_PATH}")
print(f"[setup] output: {OUTPUT_DIR}")

In [ ]:
# Cell 2: imports
import csv
import json
import random
import shutil
import time
import itertools

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoConfig, AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

SEED = 42
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[setup] device: {device}")
if device.type == "cuda":
    print(f"[setup] GPU: {torch.cuda.get_device_name(0)} | count: {torch.cuda.device_count()}")

In [ ]:
# Cell 3: load data
WINNER_TO_LABEL = {"a": 0, "b": 1, "tie": 2}
LABEL_TO_WINNER = {v: k for k, v in WINNER_TO_LABEL.items()}

df = pd.read_parquet(DATA_PATH)
print(f"[data] {len(df)} rows, {df['fold'].nunique()} folds")
print(df["winner"].value_counts(normalize=True).round(3))
print(df.columns.tolist())

In [ ]:
# Cell 4: dataset and collator
class RewardPairDataset(Dataset):
    def __init__(self, df, tokenizer, max_length):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = row["prompt_text"]

        enc_a = self.tokenizer(
            text=prompt, text_pair=row["response_a_text"],
            truncation="longest_first", max_length=self.max_length,
        )
        enc_b = self.tokenizer(
            text=prompt, text_pair=row["response_b_text"],
            truncation="longest_first", max_length=self.max_length,
        )
        return {
            "input_ids_a": enc_a["input_ids"], "attention_mask_a": enc_a["attention_mask"],
            "input_ids_b": enc_b["input_ids"], "attention_mask_b": enc_b["attention_mask"],
            "label": WINNER_TO_LABEL[row["winner"]],
        }


class RewardPairCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, batch):
        input_ids, attention_mask = [], []
        for item in batch:
            input_ids.append(item["input_ids_a"])
            input_ids.append(item["input_ids_b"])
            attention_mask.append(item["attention_mask_a"])
            attention_mask.append(item["attention_mask_b"])

        padded = self.tokenizer.pad(
            {"input_ids": input_ids, "attention_mask": attention_mask},
            padding=True, return_tensors="pt",
        )
        labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)
        return {"input_ids": padded["input_ids"], "attention_mask": padded["attention_mask"], "labels": labels}


def load_fold_split(df, fold):
    train_df = df[df["fold"] != fold]
    val_df = df[df["fold"] == fold]
    return train_df, val_df


print("[setup] dataset & collator defined")

In [ ]:
# Cell 5: reward model
class RewardModel(nn.Module):
    def __init__(self, backbone="microsoft/deberta-v3-base", pooling="cls", dropout=0.1, delta_init=0.0):
        super().__init__()
        if pooling not in ("cls", "mean"):
            raise ValueError(f"pooling must be 'cls' or 'mean', got {pooling!r}")
        self.pooling = pooling

        config = AutoConfig.from_pretrained(backbone)
        # deberta-v3-base's config.json specifies torch_dtype float16;
        # force fp32 weights so they match the fp32 reward_head below.
        # fp16 training is handled by autocast, not by half-precision weights.
        self.encoder = AutoModel.from_pretrained(backbone, config=config, torch_dtype=torch.float32)
        self.encoder.gradient_checkpointing_enable()

        self.dropout = nn.Dropout(dropout)
        self.reward_head = nn.Linear(config.hidden_size, 1)
        self.delta = nn.Parameter(torch.tensor(float(delta_init)))
        self._eps = 1e-6

    def _pool(self, last_hidden_state, attention_mask):
        if self.pooling == "cls":
            return last_hidden_state[:, 0, :]
        mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
        summed = (last_hidden_state * mask).sum(dim=1)
        counted = mask.sum(dim=1).clamp(min=1e-6)
        return summed / counted

    def encode(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self._pool(out.last_hidden_state, attention_mask)
        pooled = self.dropout(pooled)
        return self.reward_head(pooled).squeeze(-1)

    def forward(self, input_ids, attention_mask):
        rewards = self.encode(input_ids, attention_mask)  # (batch * 2,), interleaved [a0, b0, a1, b1, ...]
        r_a = rewards[0::2]
        r_b = rewards[1::2]
        d = r_a - r_b

        p_a = torch.sigmoid(d - self.delta)
        p_b = torch.sigmoid(-d - self.delta)
        p_tie = (1.0 - p_a - p_b).clamp(min=self._eps)

        class_logits = torch.stack(
            [torch.log(p_a.clamp(min=self._eps)),
             torch.log(p_b.clamp(min=self._eps)),
             torch.log(p_tie)],
            dim=1,
        )
        return {"r_a": r_a, "r_b": r_b, "d": d, "delta": self.delta,
                "p_a": p_a, "p_b": p_b, "p_tie": p_tie, "class_logits": class_logits}


print("[setup] RewardModel defined")

In [ ]:
# Cell 6: config + helpers (metrics, param groups, checkpoint I/O)
CONFIG = {
    "backbone": "microsoft/deberta-v3-base",
    "pooling": "cls",
    "max_length": 1024,
    "dropout": 0.1,
    "delta_init": 0.0,
    "epochs": 4,
    "batch_size": 4,
    "eval_batch_size": 8,
    "grad_accum_steps": 2,
    "lr": 5.0e-5,
    "head_lr": 1.0e-4,
    "tie_lr": 2.0e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.06,
    "max_grad_norm": 1.0,
    "precision": "fp16",
    "num_workers": 2,
    "log_every_n_steps": 200,
    "n_folds": 5,
    "folds_to_run": [0, 1, 2, 3, 4],
    "checkpoint_every_n_steps": 1000
}

METRICS_PATH = OUTPUT_DIR / "metrics_history.csv"
PROGRESS_PATH = OUTPUT_DIR / "progress.json"


def multiclass_log_loss(probs, labels, eps=1e-15):
    p = np.clip(probs[np.arange(len(labels)), labels], eps, 1 - eps)
    return float(-np.log(p).mean())


def build_param_groups(model, lr, head_lr, tie_lr, weight_decay):
    no_decay = ("bias", "LayerNorm.weight", "layer_norm.weight")
    encoder_decay, encoder_no_decay = [], []
    for name, param in model.encoder.named_parameters():
        (encoder_no_decay if any(nd in name for nd in no_decay) else encoder_decay).append(param)
    return [
        {"params": encoder_decay, "lr": lr, "weight_decay": weight_decay},
        {"params": encoder_no_decay, "lr": lr, "weight_decay": 0.0},
        {"params": list(model.reward_head.parameters()), "lr": head_lr, "weight_decay": 0.0},
        {"params": [model.delta], "lr": tie_lr, "weight_decay": 0.0},
    ]


def append_metrics_row(row: dict):
    is_new = not METRICS_PATH.exists()
    with open(METRICS_PATH, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if is_new:
            writer.writeheader()
        writer.writerow(row)


def load_progress() -> dict:
    if PROGRESS_PATH.exists():
        with open(PROGRESS_PATH) as f:
            return json.load(f)
    return {"completed_folds": [], "current_fold": None, "current_fold_completed_epochs": 0}


def save_progress(progress: dict):
    with open(PROGRESS_PATH, "w") as f:
        json.dump(progress, f, indent=2)


print("[setup] config + helpers defined")
print(CONFIG)

In [ ]:
# Cell 7: resume setup + checkpoint save/load
if RESUME_FROM is not None and RESUME_FROM.exists():
    print(f"[resume] copying previous progress from {RESUME_FROM} into {OUTPUT_DIR}")
    shutil.copytree(RESUME_FROM, OUTPUT_DIR, dirs_exist_ok=True)
else:
    print("[resume] no RESUME_FROM set (or path doesn't exist) — starting fresh")


def save_checkpoint(path, model, optimizer, scheduler, scaler, epoch, best_val_loss):
    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "scaler_state": scaler.state_dict() if scaler is not None else None,
        "best_val_loss": best_val_loss,
    }, path)


def load_checkpoint(path, model, optimizer, scheduler, scaler):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])
    if scaler is not None and ckpt["scaler_state"] is not None:
        scaler.load_state_dict(ckpt["scaler_state"])
    return ckpt["epoch"], ckpt["best_val_loss"]


print("[setup] resume + checkpoint helpers defined")

In [ ]:
# Cell 8 (updated): run_epoch with mid-epoch step checkpointing + resume-by-skip
def run_epoch(model, loader, optimizer, scheduler, scaler, precision, max_grad_norm,
              log_every_n_steps, train: bool, grad_accum_steps: int = 1,
              skip_batches: int = 0, step_checkpoint_path=None,
              checkpoint_every_n_steps: int = 0, epoch_idx: int = 0):
    model.train(mode=train)
    total_loss, n_batches = 0.0, 0
    all_probs, all_labels = [], []

    use_amp = precision == "fp16" and device.type == "cuda"
    autocast_dtype = torch.float16 if precision == "fp16" else torch.float32

    if train:
        optimizer.zero_grad(set_to_none=True)

    iterator = enumerate(loader)
    if skip_batches > 0:
        print(f"    [resume] skipping {skip_batches} already-processed batches")
        iterator = itertools.islice(iterator, skip_batches, None)

    for step, batch in iterator:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with torch.set_grad_enabled(train):
            with torch.autocast(device_type=device.type, dtype=autocast_dtype, enabled=use_amp):
                out = model(input_ids, attention_mask)
                loss = F.cross_entropy(out["class_logits"], labels)

        if train:
            loss_to_backward = loss / grad_accum_steps
            if use_amp:
                scaler.scale(loss_to_backward).backward()
            else:
                loss_to_backward.backward()

            if (step + 1) % grad_accum_steps == 0:
                if use_amp:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                    optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            if (checkpoint_every_n_steps and step_checkpoint_path
                    and (step + 1) % checkpoint_every_n_steps == 0):
                torch.save({
                    "epoch": epoch_idx, "next_batch_idx": step + 1,
                    "model_state": model.state_dict(), "optimizer_state": optimizer.state_dict(),
                    "scheduler_state": scheduler.state_dict(),
                    "scaler_state": scaler.state_dict() if scaler is not None else None,
                }, step_checkpoint_path)
                progress = load_progress()
                progress["mid_epoch_fold"] = None  # set by caller right after, see run_fold
                print(f"    [checkpoint] saved mid-epoch state at batch {step + 1}")

        total_loss += loss.item()
        n_batches += 1

        probs = torch.stack([out["p_a"], out["p_b"], out["p_tie"]], dim=1).detach()
        all_probs.append(probs.cpu())
        all_labels.append(labels.cpu())

        if train and log_every_n_steps and (step + 1) % log_every_n_steps == 0:
            print(f"    step {step + 1}/{len(loader)} | loss {total_loss / n_batches:.4f} | delta {model.delta.item():.4f}")

    all_probs = torch.cat(all_probs).numpy()
    all_labels = torch.cat(all_labels).numpy()
    return total_loss / max(n_batches, 1), all_probs, all_labels


print("[setup] run_epoch (with step-level checkpointing) defined")

In [ ]:
# Cell 9 (updated): run_fold with mid-epoch resume support
def run_fold(fold: int, resume_epoch: int = 0, resume_skip_batches: int = 0):
    print(f"\n=== Fold {fold} ===")
    train_df, val_df = load_fold_split(df, fold)
    print(f"[data] train {len(train_df)} rows | val {len(val_df)} rows")

    tokenizer = AutoTokenizer.from_pretrained(CONFIG["backbone"])
    train_ds = RewardPairDataset(train_df, tokenizer, CONFIG["max_length"])
    val_ds = RewardPairDataset(val_df, tokenizer, CONFIG["max_length"])
    collator = RewardPairCollator(tokenizer)

    val_loader = DataLoader(val_ds, batch_size=CONFIG["eval_batch_size"], shuffle=False,
                             collate_fn=collator, num_workers=CONFIG["num_workers"])

    model = RewardModel(CONFIG["backbone"], CONFIG["pooling"], CONFIG["dropout"], CONFIG["delta_init"]).to(device)
    optimizer = torch.optim.AdamW(build_param_groups(model, CONFIG["lr"], CONFIG["head_lr"], CONFIG["tie_lr"], CONFIG["weight_decay"]))

    n_steps_per_epoch = len(train_ds) // CONFIG["batch_size"] // CONFIG["grad_accum_steps"]
    n_steps = n_steps_per_epoch * CONFIG["epochs"]
    n_warmup = int(n_steps * CONFIG["warmup_ratio"])
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=n_warmup, num_training_steps=n_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=(CONFIG["precision"] == "fp16" and device.type == "cuda"))

    fold_dir = OUTPUT_DIR / f"fold{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = fold_dir / "checkpoint.pt"          # epoch-boundary checkpoint
    step_checkpoint_path = fold_dir / "step_checkpoint.pt"  # mid-epoch checkpoint
    best_path = fold_dir / "best.pt"

    best_val_loss = float("inf")
    start_epoch = 0
    skip_batches = 0

    if resume_epoch > 0 and checkpoint_path.exists():
        loaded_epoch, best_val_loss = load_checkpoint(checkpoint_path, model, optimizer, scheduler, scaler)
        start_epoch = loaded_epoch + 1
        print(f"[resume] fold {fold}: resuming from epoch boundary {start_epoch} (best_val_loss so far: {best_val_loss:.4f})")

    if resume_skip_batches > 0 and step_checkpoint_path.exists():
        ckpt = torch.load(step_checkpoint_path, map_location=device)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        scheduler.load_state_dict(ckpt["scheduler_state"])
        if scaler is not None and ckpt["scaler_state"] is not None:
            scaler.load_state_dict(ckpt["scaler_state"])
        start_epoch = ckpt["epoch"]
        skip_batches = ckpt["next_batch_idx"]
        print(f"[resume] fold {fold}: resuming mid-epoch {start_epoch + 1}, skipping first {skip_batches} batches")

    best_val_probs, best_val_labels = None, None

    for epoch in range(start_epoch, CONFIG["epochs"]):
        t0 = time.time()
        epoch_skip = skip_batches if epoch == start_epoch else 0

        train_gen = torch.Generator().manual_seed(1000 + fold * 100 + epoch)
        train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,
                                   collate_fn=collator, num_workers=CONFIG["num_workers"],
                                   drop_last=True, generator=train_gen)

        train_loss, _, _ = run_epoch(
            model, train_loader, optimizer, scheduler, scaler,
            CONFIG["precision"], CONFIG["max_grad_norm"], CONFIG["log_every_n_steps"], train=True,
            grad_accum_steps=CONFIG["grad_accum_steps"], skip_batches=epoch_skip,
            step_checkpoint_path=step_checkpoint_path,
            checkpoint_every_n_steps=CONFIG["checkpoint_every_n_steps"], epoch_idx=epoch,
        )
        val_loss, val_probs, val_labels = run_epoch(
            model, val_loader, None, None, None,
            CONFIG["precision"], None, None, train=False,
        )
        val_ll = multiclass_log_loss(val_probs, val_labels)
        dt = time.time() - t0

        print(f"[fold {fold}][epoch {epoch + 1}/{CONFIG['epochs']}] train_loss {train_loss:.4f} | "
              f"val_loss {val_loss:.4f} | val_log_loss {val_ll:.4f} | delta {model.delta.item():.4f} | {dt:.0f}s")

        append_metrics_row({
            "fold": fold, "epoch": epoch + 1, "train_loss": round(train_loss, 6),
            "val_loss": round(val_loss, 6), "val_log_loss": round(val_ll, 6),
            "delta": round(model.delta.item(), 6), "elapsed_sec": round(dt, 1),
        })

        if val_ll < best_val_loss:
            best_val_loss = val_ll
            best_val_probs, best_val_labels = val_probs, val_labels
            torch.save(model.state_dict(), best_path)

        save_checkpoint(checkpoint_path, model, optimizer, scheduler, scaler, epoch, best_val_loss)
        if step_checkpoint_path.exists():
            step_checkpoint_path.unlink()  # epoch finished cleanly, mid-epoch checkpoint is now stale

        progress = load_progress()
        progress["current_fold"] = fold
        progress["current_fold_completed_epochs"] = epoch + 1
        save_progress(progress)

    del model, optimizer, scheduler
    torch.cuda.empty_cache()

    return {
        "fold": fold, "val_log_loss": best_val_loss,
        "val_probs": best_val_probs, "val_labels": best_val_labels,
        "val_ids": val_df["id"].to_numpy(),
    }


print("[setup] run_fold (with mid-epoch resume) defined")

In [ ]:
# Cell 10: main loop over folds, with resume (epoch-level + mid-epoch)
progress = load_progress()
print(f"[main] progress so far: {progress}")

completed_folds = set(progress["completed_folds"])
results = []

if METRICS_PATH.exists():
    _hist = pd.read_csv(METRICS_PATH)
else:
    _hist = None

for fold in CONFIG["folds_to_run"]:
    if fold in completed_folds:
        print(f"[main] fold {fold} already completed, loading its saved val predictions")
        fold_dir = OUTPUT_DIR / f"fold{fold}"
        saved = torch.load(fold_dir / "checkpoint.pt", map_location=device)
        best_ll = saved["best_val_loss"]
        _, val_df_f = load_fold_split(df, fold)
        tok = AutoTokenizer.from_pretrained(CONFIG["backbone"])
        val_ds_f = RewardPairDataset(val_df_f, tok, CONFIG["max_length"])
        val_loader_f = DataLoader(val_ds_f, batch_size=CONFIG["eval_batch_size"], shuffle=False,
                                   collate_fn=RewardPairCollator(tok), num_workers=CONFIG["num_workers"])
        m = RewardModel(CONFIG["backbone"], CONFIG["pooling"], CONFIG["dropout"], CONFIG["delta_init"]).to(device)
        m.load_state_dict(torch.load(fold_dir / "best.pt", map_location=device))
        _, val_probs_f, val_labels_f = run_epoch(m, val_loader_f, None, None, None,
                                                   CONFIG["precision"], None, None, train=False)
        del m
        torch.cuda.empty_cache()
        results.append({"fold": fold, "val_log_loss": best_ll, "val_probs": val_probs_f,
                         "val_labels": val_labels_f, "val_ids": val_df_f["id"].to_numpy()})
        continue

    # NEW: check for a mid-epoch step checkpoint, in addition to the epoch-boundary one
    fold_dir = OUTPUT_DIR / f"fold{fold}"
    step_ckpt = fold_dir / "step_checkpoint.pt"
    resume_skip = 0
    if step_ckpt.exists():
        resume_skip = torch.load(step_ckpt, map_location="cpu")["next_batch_idx"]
        print(f"[main] fold {fold} has a mid-epoch checkpoint, will skip first {resume_skip} batches")

    resume_epoch = progress["current_fold_completed_epochs"] if progress.get("current_fold") == fold else 0
    r = run_fold(fold, resume_epoch=resume_epoch, resume_skip_batches=resume_skip)  # NEW: resume_skip_batches arg
    results.append(r)

    progress["completed_folds"] = list(set(progress["completed_folds"]) | {fold})
    progress["current_fold"] = None
    progress["current_fold_completed_epochs"] = 0
    save_progress(progress)

print("\n=== Per-fold val log loss ===")
for r in results:
    print(f"  fold {r['fold']}: {r['val_log_loss']:.4f}")